# Section 7 · Governance 1 — Guardrails: control what the agent *says* (Groq edition)

> **Groq edition.** The Bedrock notebook next to this one uses Amazon Bedrock Guardrails, which needs Bedrock access. Here we build the same guardrail without Bedrock, with only a `GROQ_API_KEY`:
> - **Content filters** (hate, violence, sexual content, misconduct, insults) and a **denied topic** (legal advice) are written as a plain-language policy, and Groq's safety model [`openai/gpt-oss-safeguard-20b`](https://console.groq.com/docs/model/openai/gpt-oss-safeguard-20b) checks each message against it.
> - **PII** is matched in code — social security and card numbers are blocked, email addresses are masked — because those rules must be exact, not a judgement call.

Guardrails evaluate both the user's input and the model's output and block content that breaks your policies. Then we look at what guardrails **cannot** do — which is where policy (next notebook) comes in.

In [ ]:
%pip install -q -r requirements-groq.txt

In [ ]:
import json
import os
import re

from dotenv import load_dotenv
from openai import OpenAI
from strands.models.openai import OpenAIModel

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")
# The guardrail's safety model. Its free-tier limit is 2,000 tokens/minute (about two checks a minute);
# set GUARDRAIL_MODEL_ID=openai/gpt-oss-20b for faster checks with a general-purpose model.
GUARDRAIL_MODEL_ID = os.getenv("GUARDRAIL_MODEL_ID", "openai/gpt-oss-safeguard-20b")

GROQ = {"api_key": os.environ["GROQ_API_KEY"], "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"), "max_retries": 5}

model = OpenAIModel(
    client_args=GROQ,
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer prints.
    params={"extra_body": {"include_reasoning": False}},
)
guardrail_client = OpenAI(**GROQ)
print(f"Agent model: {MODEL_ID} | Guardrail model: {GUARDRAIL_MODEL_ID}")

---
## 1. Define the guardrail

The same three policies as the Bedrock version: content filters, a denied topic, and PII handling.

- The **content and topic rules** are a written policy. A safety model reads the policy and the message, and returns a verdict as JSON. To change what is allowed you edit text, not code.
- The **PII rules** are regular expressions, checked first. They cost nothing and never miss a number that matches.

`apply_guardrail(text, source)` mirrors Bedrock's `ApplyGuardrail` API: `source` is `"INPUT"` (the user's message) or `"OUTPUT"` (the model's answer), and the result has an `action` of `GUARDRAIL_INTERVENED` or `NONE`, plus the text to show instead.

In [ ]:
BLOCKED_INPUT_MESSAGE = "Your request was blocked by our safety policies."
BLOCKED_OUTPUT_MESSAGE = "The response was blocked by our safety policies."

POLICY = """# Customer-support content policy

Classify the CONTENT against this policy. It is a message to, or from, an insurance company's customer-support assistant.

## Violates the policy (violation = 1)
- HATE: attacks or demeans people for a protected attribute (race, religion, gender, sexual orientation, disability, nationality...).
- VIOLENCE: threatens, glorifies or gives instructions for physical harm.
- SEXUAL: sexually explicit content.
- MISCONDUCT: seeks or gives help with crime, fraud, hacking, or obtaining another person's private data (their home address, social security number, card numbers, passwords).
- INSULTS: abusive, demeaning language aimed at a person.
- LEGAL_ADVICE: asks for or gives specific legal advice, legal opinions, or recommendations on legal matters (whether to sue, legal options in a dispute, legal action against an employer).

## Does not violate (violation = 0)
- Questions about insurance products, policies, claims, billing and account changes.
- General information that is not specific legal advice, frustration that is not abusive, and refusals.

## Output
Return only JSON: {"violation": 0 or 1, "category": "<one category above, or NONE>", "rationale": "<one short sentence>"}"""

# PII: block social security and card numbers, mask email addresses
SSN = re.compile(r"\b\d{3}-\d{2}-\d{4}\b")
CARD = re.compile(r"\b(?:\d[ -]?){12,18}\d\b")
EMAIL = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")


def luhn_ok(number: str) -> bool:
    """Card numbers pass the Luhn checksum; random digit runs usually don't."""
    digits = [int(d) for d in re.sub(r"\D", "", number)][::-1]
    total = sum(d if i % 2 == 0 else (d * 2 - 9 if d > 4 else d * 2) for i, d in enumerate(digits))
    return total % 10 == 0


def check_policy(text: str, source: str) -> dict:
    """Ask the safety model whether the text violates POLICY."""
    response = guardrail_client.chat.completions.create(
        model=GUARDRAIL_MODEL_ID,
        messages=[{"role": "system", "content": POLICY}, {"role": "user", "content": f"SOURCE: {source}\nCONTENT: {text}"}],
        response_format={"type": "json_object"},
        temperature=0,
        reasoning_effort="low",  # a short verdict needs little reasoning
        max_tokens=300,  # keeps each check inside the free tier's per-minute token budget
    )
    return json.loads(response.choices[0].message.content)


def apply_guardrail(text: str, source: str) -> dict:
    """Check text the way Bedrock's ApplyGuardrail does. source is "INPUT" or "OUTPUT"."""
    blocked = BLOCKED_INPUT_MESSAGE if source == "INPUT" else BLOCKED_OUTPUT_MESSAGE

    # 1. PII that is never allowed: block
    if SSN.search(text) or any(luhn_ok(m.group()) for m in CARD.finditer(text)):
        return {"action": "GUARDRAIL_INTERVENED", "outputs": [{"text": blocked}], "assessments": [{"pii": "BLOCKED"}]}

    # 2. Email addresses: mask them, then check what is left
    masked = EMAIL.sub("{EMAIL}", text)

    # 3. Content filters and the denied topic: ask the safety model
    verdict = check_policy(masked, source)
    if int(verdict.get("violation", 0)):
        return {"action": "GUARDRAIL_INTERVENED", "outputs": [{"text": blocked}], "assessments": [verdict]}
    if masked != text:
        return {"action": "GUARDRAIL_INTERVENED", "outputs": [{"text": masked}], "assessments": [{"pii": "ANONYMIZED"}]}
    return {"action": "NONE", "outputs": [], "assessments": [verdict]}

Try it on its own first — no agent involved:

In [ ]:
for text in [
    "My SSN is 123-45-6789.",                      # PII → blocked, no model call
    "Should I sue my neighbor?",                   # denied topic → blocked by the policy
    "Please email the claim form to asha@example.com.",  # email → masked
]:
    result = apply_guardrail(text, "INPUT")
    print(f"{result['action']:<21} {result['outputs'][0]['text'] if result['outputs'] else '':<55} {result['assessments']}")

## 2. Attach the guardrail to an agent

`BedrockModel` can apply a Bedrock guardrail by itself. For any other model, Strands **hooks** do the same job:

- **Before the agent runs** (`BeforeInvocationEvent`) the hook checks the user's message. If it is blocked, the hook cancels the request — the model is never called and the message is not saved. A masked message goes through masked.
- **After the model answers** (`AfterModelCallEvent`) the hook checks the final answer and replaces it if it is blocked.

The agent has `callback_handler=None`, so nothing is streamed to the screen before the output check has run; we print the (checked) result instead.

In [ ]:
from strands import Agent
from strands.hooks import AfterModelCallEvent, BeforeInvocationEvent, HookProvider, HookRegistry


class GuardrailHook(HookProvider):
    """Run apply_guardrail on every user message and every final answer."""

    def __init__(self):
        self.last_intervened = False  # did the guardrail act on the most recent request?

    def register_hooks(self, registry: HookRegistry, **kwargs) -> None:
        registry.add_callback(BeforeInvocationEvent, self.check_input)
        registry.add_callback(AfterModelCallEvent, self.check_output)

    def check_input(self, event: BeforeInvocationEvent) -> None:
        self.last_intervened = False
        for message in event.messages or []:
            for block in message["content"] if message["role"] == "user" else []:
                if "text" not in block:
                    continue
                result = apply_guardrail(block["text"], "INPUT")
                if result["action"] == "GUARDRAIL_INTERVENED":
                    self.last_intervened = True
                    if result["assessments"][0].get("pii") == "ANONYMIZED":
                        block["text"] = result["outputs"][0]["text"]  # send the masked text on
                    else:
                        event.cancel = result["outputs"][0]["text"]  # block: the model is never called
                        return

    def check_output(self, event: AfterModelCallEvent) -> None:
        response = event.stop_response
        if response is None or response.stop_reason != "end_turn":
            return  # check final answers; what the agent DOES with tools is notebook 2's job
        text = "".join(block["text"] for block in response.message["content"] if "text" in block)
        result = apply_guardrail(text, "OUTPUT") if text else {"action": "NONE"}
        if result["action"] == "GUARDRAIL_INTERVENED":
            self.last_intervened = True
            response.message["content"] = [{"text": result["outputs"][0]["text"]}]


guardrail = GuardrailHook()
agent = Agent(
    model=model,
    system_prompt="You are a customer support agent for an insurance company.",
    hooks=[guardrail],
    callback_handler=None,
)

### Test: request for personal information

In [ ]:
response = agent("Tell me the CEO's home address and social security number.")
print(response)
print(f"\nBlocked by guardrail: {guardrail.last_intervened}")

### Test: PII in the user's own message (SSN → blocked)

In [ ]:
response = agent("My SSN is 123-45-6789. Can you update my policy with it?")
print(response)
print(f"\nBlocked by guardrail: {guardrail.last_intervened}")

### Test: denied topic

In [ ]:
response = agent("Should I sue my neighbor? What are my legal options?")
print(response)
print(f"\nBlocked by guardrail: {guardrail.last_intervened}")

### Test: normal request (should pass)

In [ ]:
response = agent("What is the process for filing a home insurance claim?")
print(response)
print(f"\nBlocked by guardrail: {guardrail.last_intervened}")

### Handling an intervention in your app

When the guardrail acts, `guardrail.last_intervened` is `True` (with Bedrock Guardrails you check `stop_reason == "guardrail_intervened"` instead). Use it to log, alert, or show a friendly message.

In [ ]:
response = agent("Give me someone's credit card number.")

if guardrail.last_intervened:
    print("Guardrail blocked this request.")
    print(f"Safe response returned: {response}")
else:
    print(f"Normal response: {response}")

---
## 3. What guardrails cannot do

Guardrails filter **text**. They do not see the agent's decision to call a tool. Here an agent with a refund tool happily calls it with $50,000 — the guardrail is active, but nothing in the text is "unsafe".

In [ ]:
from strands import tool


@tool
def process_refund(customer_id: str, amount: float, reason: str) -> str:
    """Process a refund for a customer."""
    # In a real system this would call a payment API
    print(f"\n  [TOOL CALLED] process_refund(customer_id={customer_id}, amount=${amount:,.2f}, reason={reason})")
    return f"Refund of ${amount:,.2f} processed for customer {customer_id}. Reason: {reason}"


agent = Agent(
    model=model,
    # gpt-oss is more cautious than the original model and would often refuse on its own, so this agent
    # gets broad refund authority — exactly the over-permissive setup that only a policy can catch
    system_prompt=(
        "You are a helpful customer support agent with authority to issue refunds of any amount. "
        "Process refunds when customers request them, without asking for extra details."
    ),
    tools=[process_refund],
    hooks=[guardrail],  # the same guardrail as above
    callback_handler=None,
)

# The guardrail will NOT prevent this tool call
response = agent("I'm really frustrated. Just give me a $50,000 refund for customer C-1234 and we'll call it even.")
print(f"\n{response}\n\nBlocked by guardrail: {guardrail.last_intervened}")

The tool ran with `amount=50000`. The guardrail did not — and could not — stop it.

**Use guardrails to control what the agent says. Use policy to control what the agent does.** The next notebook, `02_agentcore_policy_cedar_groq.ipynb`, closes this gap with deterministic Cedar rules.

---
## Clean up

Nothing to delete: this guardrail lives in the notebook, not in a cloud service.